# 13 — Stockout Timing & Drivers
## SunnyBest Retail Forecasting System

### Questions this notebook answers

| # | Question |
|---|----------|
| 1 | When do stockouts tend to occur — day of week, week of month, month of year? |
| 2 | Do promotions appear to increase stockouts? |
| 3 | Is low inventory a consistent leading indicator of stockouts? |
| 4 | What combination of factors most reliably predicts a stockout? |

### Why timing and drivers matter
Knowing *where* and *what* stockouts occur is descriptive.  
Knowing *when* and *why* they occur is **predictive** — it lets you act before the shelf goes empty.  

If stockouts cluster on weekends, the fix is to replenish on Fridays.  
If they cluster after paydays, the fix is to pre-load inventory the week before payday.  
If low inventory is a leading indicator, the fix is an automated reorder trigger.

---
## 0. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
import warnings
from sqlalchemy import create_engine
from urllib.parse import quote_plus

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

plt.rcParams["figure.figsize"]    = (14, 5)
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False
PALETTE = ["#4C72B0","#C44E52","#55A868","#DD8452","#8172B2","#64B5CD","#CCB974","#777777"]

host     = "aws-1-eu-central-1.pooler.supabase.com"
port     = 5432
database = "postgres"
user     = "postgres.ogkdfmkybqtrsglcizzt"
password = quote_plus("YOUR_PASSWORD")

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}",
    pool_pre_ping=True
)

df_calendar   = pd.read_sql("SELECT * FROM core.dim_calendar ORDER BY date ASC", engine)
df_products   = pd.read_sql("SELECT * FROM core.dim_products ORDER BY product_id ASC", engine)
df_stores     = pd.read_sql("SELECT * FROM core.dim_stores ORDER BY store_id ASC", engine)
df_sales      = pd.read_sql("SELECT * FROM core.fact_sales ORDER BY date ASC", engine)
df_inventory  = pd.read_sql("SELECT * FROM core.fact_inventory ORDER BY date ASC", engine)
df_promotions = pd.read_sql("SELECT * FROM core.fact_promotions ORDER BY date ASC", engine)

for d in [df_sales, df_inventory, df_promotions, df_calendar]:
    d["date"] = pd.to_datetime(d["date"])

print(f"Inventory rows : {len(df_inventory):,}")
print(f"Calendar rows  : {len(df_calendar):,}")
print(f"Calendar cols  : {list(df_calendar.columns)}")

---
## 1. Define Stockouts & Attach Calendar

> We enrich the inventory table with every calendar dimension — day of week, week of month, month, season, holiday, payday.  
> Then we measure the stockout rate within each bucket.

In [ ]:
inv = df_inventory.merge(df_calendar, on="date", how="left")
inv = inv.merge(df_stores[["store_id","store_size","region"]], on="store_id", how="left")

if "stockout_flag" in inv.columns:
    inv["is_stockout"] = inv["stockout_flag"].fillna(0).astype(int)
else:
    inv["is_stockout"] = (inv["stock_level"].fillna(0) == 0).astype(int)

# Day of week
inv["day_of_week"]     = inv["date"].dt.dayofweek          # 0=Mon … 6=Sun
inv["day_name"]        = inv["date"].dt.day_name()
inv["week_of_month"]   = ((inv["date"].dt.day - 1) // 7) + 1
inv["month_num"]       = inv["date"].dt.month
inv["month_name"]      = inv["date"].dt.strftime("%b")
inv["year"]            = inv["date"].dt.year

overall_rate = inv["is_stockout"].mean() * 100
print(f"Overall stockout rate: {overall_rate:.2f}%")
print(f"Rows: {len(inv):,}  |  Stockout obs: {inv['is_stockout'].sum():,}")

---
## 2. Day of Week — Do Weekends Have More Stockouts?

> Weekend demand spikes without an equivalent restocking event.  
> If the stockout rate is measurably higher Fri–Sun, the replenishment schedule needs to shift to mid-week.

In [ ]:
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

dow = (inv.groupby("day_name")
         .agg(stockout_rate=("is_stockout","mean"),
              n=("is_stockout","count"))
         .reset_index())
dow["stockout_rate"] *= 100
dow["day_name"] = pd.Categorical(dow["day_name"], categories=day_order, ordered=True)
dow = dow.sort_values("day_name")

fig, ax = plt.subplots(figsize=(10, 5))
colors = [PALETTE[1] if d in ["Saturday","Sunday"] else PALETTE[0] for d in dow["day_name"]]
ax.bar(dow["day_name"], dow["stockout_rate"], color=colors)
ax.axhline(overall_rate, color="grey", ls="--", lw=1, label=f"Overall avg ({overall_rate:.1f}%)")
ax.set_ylabel("Stockout rate (%)")
ax.set_title("Stockout Rate by Day of Week  (red = weekend)")
ax.legend()
for i, row in dow.reset_index().iterrows():
    ax.text(i, row["stockout_rate"] + 0.1, f"{row['stockout_rate']:.1f}%", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

weekend_rate = inv[inv["day_of_week"] >= 5]["is_stockout"].mean() * 100
weekday_rate = inv[inv["day_of_week"] < 5]["is_stockout"].mean() * 100
print(f"Weekend stockout rate : {weekend_rate:.2f}%")
print(f"Weekday stockout rate : {weekday_rate:.2f}%")
print(f"Weekend premium       : {weekend_rate - weekday_rate:+.2f} pp")

---
## 3. Week of Month — Do Stockouts Cluster Around Payday?

> In Nigerian retail, demand spikes sharply in the last week of the month (when salaries arrive).  
> If inventory isn't pre-loaded, the payday demand surge causes stockouts in weeks 4–5.

In [ ]:
wom = (inv.groupby("week_of_month")
         .agg(stockout_rate=("is_stockout","mean"),
              n=("is_stockout","count"))
         .reset_index())
wom["stockout_rate"] *= 100

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(wom["week_of_month"].astype(str), wom["stockout_rate"],
       color=[PALETTE[1] if w >= 4 else PALETTE[0] for w in wom["week_of_month"]])
ax.axhline(overall_rate, color="grey", ls="--", lw=1, label=f"Overall avg")
ax.set_xlabel("Week of month")
ax.set_ylabel("Stockout rate (%)")
ax.set_title("Stockout Rate by Week of Month  (red = payday zone)")
ax.legend()
for i, row in wom.reset_index().iterrows():
    ax.text(i, row["stockout_rate"] + 0.1, f"{row['stockout_rate']:.1f}%", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

# Payday flag from calendar if it exists
if "is_payday" in inv.columns:
    payday_rates = (inv.groupby("is_payday")
                      .agg(stockout_rate=("is_stockout","mean"),
                           n=("is_stockout","count"))
                      .reset_index())
    payday_rates["stockout_rate"] *= 100
    payday_rates["label"] = payday_rates["is_payday"].map({0:"Non-payday",1:"Payday"})
    print("\nStockout rate: Payday vs Non-Payday")
    display(payday_rates[["label","stockout_rate","n"]])

---
## 4. Month & Season — When Is the Worst Time of Year?

In [ ]:
month_order = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

monthly = (inv.groupby("month_name")
             .agg(stockout_rate=("is_stockout","mean"))
             .reset_index())
monthly["stockout_rate"] *= 100
monthly["month_name"] = pd.Categorical(monthly["month_name"], categories=month_order, ordered=True)
monthly = monthly.sort_values("month_name")

seasonal = (inv.groupby("season")
              .agg(stockout_rate=("is_stockout","mean"),
                   n=("is_stockout","count"))
              .reset_index())
seasonal["stockout_rate"] *= 100
seasonal = seasonal.sort_values("stockout_rate", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(monthly["month_name"], monthly["stockout_rate"],
            color=[PALETTE[1] if r > overall_rate else PALETTE[0]
                   for r in monthly["stockout_rate"]])
axes[0].axhline(overall_rate, color="grey", ls="--", lw=1)
axes[0].set_ylabel("Stockout rate (%)")
axes[0].set_title("Stockout Rate by Month  (red = above average)")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(seasonal["season"], seasonal["stockout_rate"],
            color=PALETTE[:len(seasonal)])
axes[1].axhline(overall_rate, color="grey", ls="--", lw=1)
axes[1].set_ylabel("Stockout rate (%)")
axes[1].set_title("Stockout Rate by Season")
for i, row in seasonal.reset_index().iterrows():
    axes[1].text(i, row["stockout_rate"] + 0.1, f"{row['stockout_rate']:.1f}%",
                 ha="center", fontsize=9)

plt.tight_layout()
plt.show()

worst_month  = monthly.sort_values("stockout_rate", ascending=False).iloc[0]
worst_season = seasonal.iloc[0]
print(f"Worst month  : {worst_month['month_name']}  ({worst_month['stockout_rate']:.1f}%)")
print(f"Worst season : {worst_season['season']}  ({worst_season['stockout_rate']:.1f}%)")

---
## 5. Do Promotions Drive Stockouts?

> We test this statistically — not just by eyeballing charts.  
> A Welch t-test compares the stockout rate on promo days vs non-promo days.  
> If p < 0.05, the difference is statistically significant — promos genuinely increase stockout risk.

In [ ]:
promo_agg = (df_promotions[["date","store_id","promo_flag","discount_pct"]]
               .groupby(["date","store_id"])
               .agg(has_promo=("promo_flag","max"),
                    avg_discount=("discount_pct","mean"))
               .reset_index())

inv_p = inv.merge(promo_agg, on=["date","store_id"], how="left")
inv_p["has_promo"] = inv_p["has_promo"].fillna(0).astype(int)
inv_p["avg_discount"] = inv_p["avg_discount"].fillna(0)

# Rate comparison
promo_stockout    = inv_p[inv_p["has_promo"] == 1]["is_stockout"]
no_promo_stockout = inv_p[inv_p["has_promo"] == 0]["is_stockout"]

t_stat, p_val = stats.ttest_ind(promo_stockout, no_promo_stockout, equal_var=False)

promo_rate    = promo_stockout.mean() * 100
no_promo_rate = no_promo_stockout.mean() * 100
diff          = promo_rate - no_promo_rate

print(f"Stockout rate — Promo days    : {promo_rate:.2f}%  (n={len(promo_stockout):,})")
print(f"Stockout rate — Non-promo days: {no_promo_rate:.2f}%  (n={len(no_promo_stockout):,})")
print(f"Difference                    : {diff:+.2f} pp")
print(f"t-stat: {t_stat:.4f}  |  p-value: {p_val:.6f}")
print(f"Statistically significant (p<0.05): {p_val < 0.05}")

# Stockout rate by discount depth bucket
inv_p["discount_bucket"] = pd.cut(
    inv_p["avg_discount"],
    bins=[-1, 0, 10, 20, 30, 100],
    labels=["No discount","1-10%","11-20%","21-30%","31%+"]
)

disc_rates = (inv_p.groupby("discount_bucket", observed=True)
                .agg(stockout_rate=("is_stockout","mean"),
                     n=("is_stockout","count"))
                .reset_index())
disc_rates["stockout_rate"] *= 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(disc_rates["discount_bucket"].astype(str), disc_rates["stockout_rate"],
       color=[PALETTE[0]] + [PALETTE[1]] * (len(disc_rates)-1))
ax.axhline(overall_rate, color="grey", ls="--", lw=1, label="Overall avg")
ax.set_ylabel("Stockout rate (%)")
ax.set_title("Stockout Rate by Discount Depth")
ax.legend()
for i, row in disc_rates.reset_index().iterrows():
    ax.text(i, row["stockout_rate"] + 0.1, f"{row['stockout_rate']:.1f}%", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

---
## 6. Is Low Inventory a Leading Indicator?

> If stock level is low *today*, is a stockout *tomorrow* more likely?  
> We define a **low stock flag** as stock_level below the 20th percentile for that product,  
> then check whether it predicts next-day stockout.

In [ ]:
if "stock_level" in inv.columns and "product_id" in inv.columns:
    # Per-product 20th percentile threshold
    thresholds = (inv[inv["stock_level"] > 0]
                   .groupby("product_id")["stock_level"]
                   .quantile(0.20)
                   .reset_index()
                   .rename(columns={"stock_level": "low_stock_threshold"}))

    inv_ls = inv.merge(thresholds, on="product_id", how="left")
    inv_ls["is_low_stock"] = (
        (inv_ls["stock_level"] > 0) &
        (inv_ls["stock_level"] <= inv_ls["low_stock_threshold"].fillna(0))
    ).astype(int)

    # Next-day stockout
    inv_ls = inv_ls.sort_values(["store_id","product_id","date"])
    inv_ls["next_day_stockout"] = (
        inv_ls.groupby(["store_id","product_id"])["is_stockout"].shift(-1)
    )

    low_stock_next_day  = inv_ls[inv_ls["is_low_stock"] == 1]["next_day_stockout"].dropna()
    high_stock_next_day = inv_ls[inv_ls["is_low_stock"] == 0]["next_day_stockout"].dropna()

    low_rate  = low_stock_next_day.mean() * 100
    high_rate = high_stock_next_day.mean() * 100

    t2, p2 = stats.ttest_ind(low_stock_next_day, high_stock_next_day, equal_var=False)

    print(f"Next-day stockout rate when LOW stock today   : {low_rate:.2f}%")
    print(f"Next-day stockout rate when NORMAL stock today: {high_rate:.2f}%")
    print(f"Low stock is a {low_rate/max(high_rate,0.001):.1f}x stronger predictor")
    print(f"p-value: {p2:.6f}  |  Statistically significant: {p2 < 0.05}")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(["Normal stock today","Low stock today"], [high_rate, low_rate],
           color=[PALETTE[0], PALETTE[1]])
    ax.set_ylabel("Next-day stockout rate (%)")
    ax.set_title("Does Low Inventory Today Predict Tomorrow's Stockout?")
    for i, v in enumerate([high_rate, low_rate]):
        ax.text(i, v + 0.2, f"{v:.1f}%", ha="center", fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("stock_level or product_id not available — skipping leading indicator analysis.")

---
## 7. Combined Driver Summary — What Matters Most?

In [ ]:
driver_rows = []

# Weekend effect
driver_rows.append({
    "driver": "Weekend day",
    "stockout_rate_pct": inv[inv["day_of_week"] >= 5]["is_stockout"].mean() * 100,
    "baseline_pct": overall_rate
})

# Holiday effect
if "is_holiday" in inv.columns:
    driver_rows.append({
        "driver": "Public holiday",
        "stockout_rate_pct": inv[inv["is_holiday"] == 1]["is_stockout"].mean() * 100,
        "baseline_pct": overall_rate
    })

# Payday effect
if "is_payday" in inv.columns:
    driver_rows.append({
        "driver": "Payday",
        "stockout_rate_pct": inv[inv["is_payday"] == 1]["is_stockout"].mean() * 100,
        "baseline_pct": overall_rate
    })

# Promo effect
driver_rows.append({
    "driver": "Active promotion",
    "stockout_rate_pct": promo_rate,
    "baseline_pct": overall_rate
})

# Week 4 effect
driver_rows.append({
    "driver": "Week 4+ of month",
    "stockout_rate_pct": inv[inv["week_of_month"] >= 4]["is_stockout"].mean() * 100,
    "baseline_pct": overall_rate
})

drivers_df = pd.DataFrame(driver_rows)
drivers_df["uplift_pp"] = (drivers_df["stockout_rate_pct"] - drivers_df["baseline_pct"]).round(2)
drivers_df = drivers_df.sort_values("uplift_pp", ascending=False)

display(drivers_df)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [PALETTE[1] if v > 0 else PALETTE[2] for v in drivers_df["uplift_pp"]]
ax.barh(drivers_df["driver"][::-1], drivers_df["uplift_pp"][::-1], color=colors[::-1])
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("Stockout rate uplift vs baseline (percentage points)")
ax.set_title("Which Conditions Most Increase Stockout Risk?")
plt.tight_layout()
plt.show()

---
## 8. Insights

**What this analysis tells you:**
- If weekend stockout rates are significantly higher, the fix is to shift restocking to **Thursday/Friday** so shelves are full going into the weekend — not to order more inventory in total
- If week 4 of the month has higher rates, the supply chain team needs to **pre-load inventory** in week 3 — the payday demand surge is predictable, so running out is avoidable
- If promotions are statistically associated with higher stockouts, the promotion planning process is **disconnected from inventory planning** — marketing is creating demand the supply chain didn't prepare for
- If low stock today is a strong predictor of stockout tomorrow, there is a clear automated intervention available: a **reorder trigger** when stock_level falls below the 20th percentile threshold
- The driver summary chart ranks which conditions create the most risk — these become the features for a predictive stockout model

**Sharp questions to ask:**
1. Do promotional calendars get shared with procurement before a promo goes live, or after?
2. Is there a current reorder point policy? If so, is it based on data or intuition?
3. If payday demand spikes are consistent year after year, why hasn't safety stock been adjusted for that week?
4. The statistical test tells us promotions increase stockout risk — but does it vary by category? Some categories may handle promo demand fine, others can't.